In [ ]:
import zipfile
import astropy.io.fits as fits
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np

### Read the entire header of your fits file ###

In [4]:
def get_fits_headers_from_zip(zip_path):
    """
    Reads the FITS headers of all FITS files within a zip archive
    without extracting the entire archive.

    Args:
        zip_path (str): The path to the zip file.

    Returns:
        dict: A dictionary where keys are filenames and values are the
              FITS headers (astropy.io.fits.Header objects).
    """
    headers = {}
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_file:
            # Get a list of all files in the zip archive
            file_list = zip_file.namelist()

            # Iterate through each file in the archive
            for filename in file_list:
                # Check if the file is a FITS file
                if filename.endswith(('.fits', '.fit')):
                    try:
                        # Open the FITS file within the zip archive
                        with zip_file.open(filename) as fits_file:
                            # Read the FITS header from the file
                            # using astropy.io.fits
                            header = fits.getheader(fits_file)
                            headers[filename] = header
                    except Exception as e:
                        print(f"Could not read header for {filename}: {e}")
    except FileNotFoundError:
        print(f"Error: The file {zip_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return headers

In [ ]:
zip_file_path = 'D:/Pavlicek, Emma/G191B2B/blue.zip'

In [ ]:
zip_file_path = 'D:/Pavlicek, Emma/G191B2B/blue.zip'
all_headers = get_fits_headers_from_zip(zip_file_path)

#Now you can inspect the headers to find the observation mode and other relevant information.
for filename, header in all_headers.items():
    print(f"--- Headers for {filename} ---")
    #Note: A common FITS keyword for observation mode is 'OBSTYPE' or 'MODE'
    if 'OBSTYPE' in header:
        print(f"Observation Type: {header['OBSTYPE']}")
    elif 'MODE' in header:
        print(f"Observation Mode: {header['MODE']}")
    else:
        print("Observation mode header not found.")
    print("-" * 20)

--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_004_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_001_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_006_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_008_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_007_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_002_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_003_BFR.fits ---
Observation mode header not found.
--------------------
--- Headers for blue/sc_G191-B2B_dual_gra

### If you don't know the observation mode keyword: ###

In [13]:
def find_observation_mode_keyword(zip_path, filename_to_inspect):
    """
    Prints the full FITS header for a specified file within a zip archive.
    """
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_file:
            if filename_to_inspect in zip_file.namelist():
                with zip_file.open(filename_to_inspect) as fits_file:
                    header = fits.getheader(fits_file)
                    print(f"--- Full Header for {filename_to_inspect} ---")
                    print(header)
                    print("-" * 50)
            else:
                print(f"Error: {filename_to_inspect} not found in the zip archive.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Pick one of the filenames from output to inspect
file_to_check = 'G191B2B/blue/mods1B/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_001_BFR.fits'

# Run the function to print the full header
find_observation_mode_keyword(zip_file_path, file_to_check)

Error: G191B2B/blue/mods1B/sc_G191-B2B_dual_grating_LS60x5_G400L_Dual_001_BFR.fits not found in the zip archive.


### Use the mode to categorize the fits by the spectrograph they were taken with ###

This is important because the two spectrographs have different sensitivity curves and should be calibrated accordingly.

In [7]:
def categorize_fits_by_instrument(zip_path):
    """
    Categorizes FITS files within a zip archive based on their INSTRUME
    header keyword.

    Args:
        zip_path (str): The path to the zip file.

    Returns:
        dict: A dictionary with instrument names as keys and lists of
              filenames as values.
    """
    categorized_files = {}
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_file:
            file_list = zip_file.namelist()

            for filename in file_list:
                if filename.endswith(('.fits', '.fit')):
                    try:
                        with zip_file.open(filename) as fits_file:
                            header = fits.getheader(fits_file)
                            # Use 'INSTRUME' to categorize the files
                            if 'INSTRUME' in header:
                                instrument_name = header['INSTRUME'].strip()
                                if instrument_name not in categorized_files:
                                    categorized_files[instrument_name] = []
                                categorized_files[instrument_name].append(filename)
                            else:
                                if 'UNCATEGORIZED' not in categorized_files:
                                    categorized_files['UNCATEGORIZED'] = []
                                categorized_files['UNCATEGORIZED'].append(filename)
                    except Exception as e:
                        print(f"Could not read header for {filename}: {e}")
    except FileNotFoundError:
        print(f"Error: The file {zip_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return categorized_files

In [ ]:
file_categories = categorize_fits_by_instrument(zip_file_path)

print("File Categories by Instrument:")
for instrument, files in file_categories.items():
    print(f"\n--- {instrument} files ---")
    for f in files:
        print(f)

File Categories by Instrument:

--- MODS1R files ---
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_025_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_024_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_026_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_023_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_030_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_027_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_028_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_029_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_031_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_032_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_033_BFCR.fits

--- MODS2R files ---
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_034_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_035_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_036_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_037_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_038_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_039_BFCR.fits
red/sc_SDSS1411_LS5x60x1.0_G670L_Dual_040

### This will help you find all differences in the fits headers, e.g., to identify all differences in the instrumental settings ###

In [15]:
def find_header_differences(zip_path):
    """
    Finds and prints header keyword differences between the first two FITS files
    in a zip archive.

    Args:
        zip_path (str): The path to the zip file.
    """
    file_headers = {}
    fits_files = []

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_file:
            # Get a list of all FITS files in the archive
            for filename in zip_file.namelist():
                if filename.endswith(('.fits', '.fit')):
                    fits_files.append(filename)

            if len(fits_files) < 2:
                print("Error: Not enough FITS files in the archive to compare.")
                return

            # Read the headers of the first two FITS files
            with zip_file.open(fits_files[0]) as file1:
                header1 = fits.getheader(file1)
                file_headers[fits_files[0]] = header1

            with zip_file.open(fits_files[1]) as file2:
                header2 = fits.getheader(file2)
                file_headers[fits_files[1]] = header2

            # Find and print the differences
            keys1 = set(header1.keys())
            keys2 = set(header2.keys())
            all_keys = keys1.union(keys2)

            print(f"--- Differences between {fits_files[0]} and {fits_files[1]} ---")
            found_diff = False
            for key in sorted(all_keys):
                if key in header1 and key in header2:
                    if header1[key] != header2[key]:
                        print(f"Key: {key}")
                        print(f"  {fits_files[0]}: {header1[key]}")
                        print(f"  {fits_files[1]}: {header2[key]}")
                        print("-" * 20)
                        found_diff = True
                elif key in header1 and key not in header2:
                    print(f"Key: {key} is only in {fits_files[0]}")
                    found_diff = True
                elif key in header2 and key not in header1:
                    print(f"Key: {key} is only in {fits_files[1]}")
                    found_diff = True

            if not found_diff:
                print("No differences found in the headers.")

    except FileNotFoundError:
        print(f"Error: The file {zip_path} was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
find_header_differences(zip_file_path)

### This allows you to get the exposure time and the d_lambda, which is important to the individual calibration of the fits ###

In [ ]:
def try_get_exptime_and_dlambda(fits_path):
    with fits.open(fits_path) as hdul:
        #exposure time: common keywords (primary or SCI extension)
        hdr_primary = hdul[0].header
        exptime = None
        for hdr in (hdr_primary,):
            for k in ("EXPTIME","EXPOSURE","TEXPTIME","HIERARCH EL EXPTIME"):
                if k in hdr:
                    exptime = float(hdr[k])
                    break
            if exptime is not None:
                break

        #Try to find a wavelength array in any HDU (data or columns)
        wave = None
        for hdu in hdul:
            #If the HDU is a BinTable or Table that has a "wavelength" or "wave" column
            if getattr(hdu, 'columns', None) is not None:
                cols = [c.name.lower() for c in hdu.columns]
                for name in ("wavelength","lambda","wave","wavelength_a","wav"):
                    if name in cols:
                        wave = hdu.data[name]
                        break
            #If image HDU but already contains a wavelength vector in header keywords, skip here
            if wave is not None:
                break

        #If no explicit wavelength array, try to reconstruct from spectral WCS
        if wave is None:
            #Try WCS on each HDU
            for hdu in hdul:
                try:
                    w = WCS(hdu.header)
                    #find spectral axis in WCS: w.wcs.spec is axis index or check ctype
                    ctypes = [c.strip().upper() for c in hdu.header.get('CTYPE1','').split()]
                    #attempt to construct a 1D wave from WCS if NAXIS1 present
                    if 'CRVAL1' in hdu.header and 'CRPIX1' in hdu.header and ('CDELT1' in hdu.header or 'CD1_1' in hdu.header):
                        n = int(hdu.header.get('NAXIS1', 0))
                        crval = float(hdu.header['CRVAL1'])
                        crpix = float(hdu.header['CRPIX1'])
                        step = float(hdu.header.get('CDELT1', hdu.header.get('CD1_1', np.nan)))
                        pix = np.arange(1, n+1, dtype=float)
                        wave = crval + (pix - crpix) * step
                        break
                except Exception:
                    continue

        #If still not found, some pipelines store the 1D spectra in separate files (.h5 or table extension).
            #At that point, you should open the 1D extracted file (or .h5) and compute median(diff(wave)).

        dlambda = None
        if wave is not None and len(wave) > 1:
            dlambda = float(np.nanmedian(np.diff(wave)))
        else:
            #fallback: try reading CDELT-like keys in the primary header
            for k in ("CDELT1","CD1_1","LBT PRO OPT DIS X_1_0","LBT PRO CRV MOD_1_0_0"):
                if k in hdr_primary:
                    try:
                        dlambda = float(hdr_primary[k])
                        break
                    except Exception:
                        pass

        return exptime, dlambda, wave

#Example:
fits_path = "D:/Pavlicek, Emma/WD1202/red/MODS1R/sc_WD1032_LS5x60x1.0_G670L_Dual_005_BFCR.fits"
exptime, dlambda, wave = try_get_exptime_and_dlambda(fits_path)
print("EXPTIME:", exptime)
print("Δλ (median diff):", dlambda)
if wave is not None:
    print("wavelength array found; sample:", wave[:5], "...", "len=", len(wave))
else:
    print("No wavelength array found in this HDU; check the extracted 1D spectrum file (.h5 or table extension).")


EXPTIME: 600.0
Δλ (median diff): -1.7083333347045482e-05
wavelength array found; sample: [158.73136332 158.73134623 158.73132915 158.73131207 158.73129498] ... len= 3088


### If the d_lambda in the fits is unreliable or is not able to be located, you can calculate it manually ###

In [ ]:
import xarray as xr

# Grab wavelength array
uncal_ds = xr.open_dataset("D:/Pavlicek, Emma/G191B2B/red/red_mods2r_spectra.h5", engine="h5netcdf")
wave = uncal_ds["wavelength"].values   # shape (M,)

#Compute median Δλ
dlambda = np.median(np.diff(wave))

print(f"Δλ (median pixel dispersion): {dlambda:.4f} Å")
print(f"λ range: {wave.min():.1f} – {wave.max():.1f} Å, Npix = {len(wave)}")

Δλ (median pixel dispersion): 0.8501 Å
λ range: 5400.0 – 10000.0 Å, Npix = 5412


### Here you can identify the units of the uncalibrated data ###

In [ ]:
try:
    hdul = fits.open('D:/Pavlicek, Emma/G191B2B/red/mods2r/sc_G191-B2B_dual_grating_LS60x5_G670L_Dual_005_BFR.fits')
    #Access the header of the primary data unit (HDU)
    header = hdul[0].header
except FileNotFoundError:
    print("Error: File not found.")
    #You can add more detailed error handling here
finally:
    hdul.close()

if 'BUNIT' in header:
    units = header['BUNIT']
    print(f"The units of the data are: {units}")
else:
    print("BUNIT keyword not found in the header. The units are not explicitly defined.")

if 'GAIN' in header:
    gain = header["GAIN"]
    print(f'The gain of the data is: {gain}')
else:
    None

The units of the data are: ADU


### Another way, perhaps more accessible, to read the entire fits header ###

In [ ]:
with fits.open('D:/Pavlicek, Emma/SDSS1411/Blue/MODS1B/sc_SDSS1411_LS5x60x1.0_G400L_Dual_025_BFR.fits') as hdul:

    header = hdul[0].header

    print(repr(header))

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axes                            
NAXIS1  =                 3088 / length of data axis 1                          
NAXIS2  =                 8192 / length of data axis 2                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
DATE    = '2025-06-08T02:47:01' / file creation date (YYYY-MM-DDThh:mm:ss UT)   
HIERARCH EL DVD NAME = 'Mods Archive'                                           
HIERARCH EL ORIG FILENAME = 'mods1b.20250227.0027'                              
HIERARCH EL EXPTIME =     600.                                                  
HIERARCH EL DATE OBS = 17406